### Humanoid

successor of Robot


In [1]:
import threading
from enum import Enum, auto
from time import sleep
from random import randint
from math import isclose, cos, sin, radians
from typing import Dict, List

In [ ]:
# Status :Enum

class Status(Enum):
    # belongs in the battery class but typing won't allow  self.Status as a type
    CHARGING = auto()
    NOT_CHARGING = auto()

    # battery health
    GOOD = auto()

    # humanoid Status will move them elsewhere
    ON = auto()

In [ ]:
# Battery


class Battery:
    def __init__(self, percentage: int = 0):
        self.percentage: float = percentage
        self.status: Status = Status.NOT_CHARGING
        self.charging_time: int = 60  # in seconds

        self.health: Status = (
            Status.GOOD
        )  # "good"|"bad" lol weather or not, to slow the charging (add inaccuracies to charging time)

    def _mod_battery(self, percentage: int = 1) -> bool:
        if (self.percentage + percentage) in range(0, 100 + 1):
            self.percentage = +percentage
            self.status = Status.NOT_CHARGING
            return True
        return False

    def charge(self, charge_length: int) -> bool:
        # charge_length: how long has it been charging or do you want to charge

        # really would like to instead have a method connect-charger that just runs in the bg and and ticks at time intervals updating percentage if still charging stops when t.cancel() is called or something closer to actual charging
        self.status = Status.CHARGING
        timer = threading.Timer(
            interval=charge_length,
            function=self._mod_battery,
            kwargs={"percentage": (charge_length / self.charging_time) * 100},
        )
        # guess a better way would be to say how long a full charge takes then define a function to map the appropriate percentage based on the interval the length of our current timer (lol bette)
        timer.start()
        return True

    def use_up(self, cost: float):
        # based on how it goes was thinking of using time to use up battery as the action is being performed at the rate of cost
        if (self.percentage - cost) >= 0:
            self.percentage -= cost
            return True
        return False

    def __str__(self):
        return f"{self.status} :{self.percentage}"

In [ ]:
# test : battery charging and use

me = Battery()
print(me.status, " ; ", me.percentage)  # initial

me.charge(3)
print(me.status, " ; ", me.percentage)  # action

sleep(4)
print(me.status, " ; ", me.percentage)  # results

me.use_up(5)
print(me.status, " ; ", me.percentage)  # results 1

me.use_up(5)

In [2]:
# Angle

class Angle:
    def __init__(self, theta:float=0):
        self.theta = theta

    @property
    def theta(self):
        return self._theta

    @theta.setter
    def theta(self, theta:float):
        self._theta = theta % 360

    def in_radians(self):
        return radians(self.theta)

    def _turn(self, beta: float =90, is_clockwise: bool=True):
        self.theta = (self.theta + (1 if is_clockwise else -1) * beta) % 360
        return True

    def clockwise(self, beta: float= 90):
        return self._turn(beta, True)

    def anticlockwise(self, beta:float = 90):
        return self._turn(beta, False)

In [ ]:
# Test : Angle turn test

a = Angle()

print(a.theta)

# print(a.clockwise())
# print(a.theta)
# print(a.clockwise())
# print(a.theta)
# print(a.clockwise())
# print(a.theta)
# print(a.clockwise())
# print(a.theta)
# print(a.clockwise())
# print(a.theta)

# print(a.anticlockwise())
# print(a.theta)
# print(a.anticlockwise())
# print(a.theta)
# print(a.anticlockwise())
# print(a.theta)
# print(a.anticlockwise())
# print(a.theta)
# print(a.anticlockwise())
# print(a.theta)

# Test : theta assignment

a.theta = 90
print(a.theta)

a.theta = 360
print(a.theta)

a.theta = 361
print(a.theta)

a = Angle(0)
print(a.theta)

a = Angle(361)
print(a.theta)


In [20]:
# Coord

class Coord:
    def __init__(self, x: float = 0, y: float = 0):
        self.x: float = x
        self.y: float = y

    @property
    def x(self):
        return self._x

    @x.setter
    def x(self, a:float):
        self._x = a

    @property
    def y(self):
        return self._y
    
    @y.setter
    def y(self, b:float):
        self._y = b

    def __eq__(self, other: object) -> bool:
        """Check equality of two coordinates."""
        if not isinstance(other, Coord):
            return NotImplemented
        return isclose(self.x, other.x, abs_tol=1e-9) and isclose(
            self.y, other.y, abs_tol=1e-9
        )

    def __hash__(self) -> int:
        """Return a hash of the angle for hashable collections."""
        return hash((self.x, self.y))

    def __repr__(self):
        return f"Coord(x={self.x}, y={self.y})"

In [32]:
# Point

class Point:
    def __init__(self, x:float, y:float, theta:float):
        self._angle:Angle = Angle(theta)
        self._coord:Coord = Coord(x, y)

    @property
    def theta(self):
        return self._angle.theta

    @theta.setter
    def theta(self, beta:float):
        self._angle.theta = beta

    @property
    def x(self):
        return self._coord.x

    @property
    def y(self):
        return self._coord.y

    @x.setter
    def x(self, a:float):
        self._coord.x = a

    @y.setter
    def y(self, b:float):
        self._coord.y = b

    def clockwise(self):
        return self._angle.clockwise()

    def anticlockwise(self):
        return self._angle.anticlockwise()

In [21]:
# Test : Point

p = Point(0, 0, 361)

print(p.theta)
print(p.x)
print(p.y)

p.theta = 362
print(p.theta)

p.x = 10
p.y = 10

print(p.x)
print(p.y)



1
0
0
2
10
10


In [38]:
# body

class Body:
    def __init__(self):
        self.head:Point = Point(0, 0, 360)
        self.tail:Point = self.head  # f it start as a point, then stretch, snake

    def move(self, length:float=1):
        # not really but im tired so..., really i think im expected to calc the new position, find the x component to add to current x with length*cos(theta)   steps * cos(rad_angle)
        self.tail = self.head

        new:Point = Point(self.head.x + length*cos(radians(self.head.theta)),
                           self.head.y + length*sin(radians(self.head.theta)), 
                           self.head.theta)
        self.head = new

    def clockwise(self):
        return self.head.clockwise()

    def anticlockwise(self):
        return self.head.anticlockwise()

In [42]:
# Test : moving body

b = Body()
print(b.head.x, b.head.y)
print(b.tail.x, b.tail.y, end='\n\n')
b.move()
print(b.head.x, b.head.y)
print(b.tail.x, b.tail.y, end='\n\n')
b.move()
print(b.head.x, b.head.y)
print(b.tail.x, b.tail.y, end='\n\n')
b.move()
print(b.head.x, b.head.y)
print(b.tail.x, b.tail.y, end='\n\n')
b.move()
print(b.head.x, b.head.y)
print(b.tail.x, b.tail.y, end='\n\n\n\n\n\n')


b = Body()
print(b.head.x, b.head.y)
print(b.tail.x, b.tail.y, end='\n\n\n\n\n\n')

b.clockwise()
print(b.head.x, b.head.y)
print(b.tail.x, b.tail.y, end='\n\n\n\n\n\n')

b.move()
print(b.head.x, b.head.y)
print(b.tail.x, b.tail.y, end='\n\n')

b.move()
print(b.head.x, b.head.y)
print(b.tail.x, b.tail.y, end='\n\n')

b.move()
print(b.head.x, b.head.y)
print(b.tail.x, b.tail.y, end='\n\n')

b.move()
print(b.head.x, b.head.y)
print(b.tail.x, b.tail.y, end='\n\n')


0 0
0 0

1.0 0.0
0 0

2.0 0.0
1.0 0.0

3.0 0.0
2.0 0.0

4.0 0.0
3.0 0.0





0 0
0 0





0 0
0 0





6.123233995736766e-17 1.0
0 0

1.2246467991473532e-16 2.0
6.123233995736766e-17 1.0

1.8369701987210297e-16 3.0
1.2246467991473532e-16 2.0

2.4492935982947064e-16 4.0
1.8369701987210297e-16 3.0



In [ ]:
# Humanoid

class Humanoid:
    def __init__(self):
        self.battery: Battery = Battery()
        self.status: Status = Status.ON
        self.body: List[Coord] = Body()

    def charge_battery(self, charge_length: int):
        self.status = Status.CHARGING
        result = self.battery.charge(charge_length)
        return result

    def __str__(self):
        return f"{self.status} : {self.battery})"

    def __repr__(self):
        return f"Humanoid({self.battery})"

In [ ]:
# test : humanoid

me = Humanoid()

print(me.battery.percentage, me.status, sep=" : ")  # initial

me.charge_battery(3)
print(me.battery.percentage, me.status, sep=" : ")  # action

sleep(4)
print(me.battery.percentage, me.status, sep=" : ")  # results

In [ ]:
# world

class World:
    def __init__(self):
        # valid coords x : -50 -> 50 ; y : -50 -> 50
        self.x_range = range(-5, 5+1)
        self.y_range = range(-5, 5+1)

        self.humanoids: Dict[Coord, Humanoid] = {}

    def spawn_humanoid(self):
        coord = Coord(randint(self.x_range.start, self.x_range.stop-1), y=randint(self.y_range.start, self.y_range.stop-1))

        if self.humanoids.get(coord) is None:
            self.humanoids[coord] = Humanoid()
            return True
        return False

In [ ]:
# test : world spawn

w = World()

print(w.humanoids)  # initial

print(w.spawn_humanoid())  # action
print(w.humanoids)
